In [5]:
import dotenv
from agents import Agent, Runner,function_tool
import requests
import os
import openai
from openai.types.chat import ChatCompletionMessage


dotenv.load_dotenv()

True

In [6]:

def get_popular_movies():
    """인기영화 정보를 가져오는 api 호출 함수입니다.
    모든 영화 정보 리스트를 가져오기 위해 사용합니다.
    """
    return requests.get("https://nomad-movies.nomadcoders.workers.dev/movies").json()


def get_movie_details(id:int):
    """id 에 해당하는 영화의 디테일한 정보를 조회하기 위해 사용합니다.
    Args:
        id: 영화 아이디
    """
    return requests.get(f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}").json()


def get_movie_similar(id:int):
    """id 에 해당하는 영화와 비슷한 영화를 리스팅 합니다.
    Args:
        id: 영화 아이디
    """
    return requests.get(f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}/similar").json()
    
FUNCTION_MAP = {
    "get_popular_movies":get_popular_movies,
    "get_movie_details":get_movie_details,
    "get_movie_similar":get_movie_similar
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "인기영화 정보를 가져오는 api 호출 함수입니다. 모든 영화 정보 리스트를 가져오기 위해 사용합니다.",
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "id 에 해당하는 영화의 디테일한 정보를 조회하기 위해 사용합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "영화 아이디",
                    },
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_similar",
            "description": "id 에 해당하는 영화와 비슷한 영화를 리스팅 합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "영화 아이디",
                    },
                },
                "required": ["id"],
            },
        },
    },
]


In [16]:
import json


messages = []
client = openai.OpenAI()

def call_ai():
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOLS,
        )

        
        process_ai_response(response.choices[0].message)

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )
        tool_list = [f"{tool_call.function.name}({tool_call.function.arguments}) 호출" for tool_call in message.tool_calls]
        print(f"Agent: {tool_list}")

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            messages.append(
                {
                    "role":"tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": json.dumps(result, ensure_ascii=False) if not isinstance(result, str) else result
                }
            )
        call_ai()
    else:
        messages.append({"role":"assistant", "content": message.content})
        print(f"Agent: {message.content}")
        print("")





      


while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()



User: 지금 인기 있는 영화 알려줘
Agent: ['get_popular_movies({}) 호출']
Agent: 현재 인기 있는 영화 목록입니다:

1. **Mercy**
   - **개요**: 가까운 미래, 한 형사가 아내를 살해한 혐의로 재판을 받고 있다. 그는 자신의 무죄를 입증하기 위해 90분의 시간을 갖고, 그를 한때 지지했던 첨단 AI 판사와 싸워야 한다.
   - **개봉일**: 2026-01-20
   - **평균 평점**: 7.1
   - ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **Shelter**
   - **개요**: 자발적인 유배 생활을 하던 남자가 폭풍에서 어린 소녀를 구하면서, 그의 은둔 생활을 깨뜨리게 된다.
   - **개봉일**: 2026-01-28
   - **평균 평점**: 6.99
   - ![Shelter](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)

3. **The Orphans**
   - **개요**: 어린 시절의 친구가 불행해진 두 사람. 그들의 첫사랑이 의문의 사고로 죽은 후, 17세 여자가 복수를 떠나면서 벌어지는 이야기.
   - **개봉일**: 2025-08-20
   - **평균 평점**: 6.14
   - ![The Orphans](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **The Bluff**
   - **개요**: 한 섬에서 평화로운 삶을 살던 전직 해적이 복수를 다짐한 전 선장이 돌아오면서 가족을 지키기 위해 과거의 끔찍한 기억과 맞서게 된다.
   - **개봉일**: 2026-02-17
   - **평균 평점**: 5.56
   - ![The Bluff](https://image.tmdb.org/t/p/w780/sojEzvfxR2DBcDSJ